# Satellite Streak Spectra by Shot — HETDEX PDR1

Aggregates all SAT-masked spaxels **per shotid** (across all IFUs in the shot),
separates distinct satellite passes within a shot by on-sky PA clustering,
and computes per-streak:

- Summed spectrum and propagated errors
- SDSS *g*-band AB magnitude (via speclite)
- Surface brightness (mag arcsec⁻²)
- Number of spaxels and IFUs used
- Flux-weighted sky centroid and median streak PA
- Mean in-band S/N

Parallelised with joblib over shotids.

**Output FITS structure**
| HDU | Name | Content |
|-----|------|---------|
| 0 | PRIMARY | metadata header |
| 1 | INFO | BinTableHDU — one row per streak (scalar quantities) |
| 2 | SPECTRA | ImageHDU (n_streaks, nwave) — summed flux erg/s/cm²/Å |
| 3 | ERRORS | ImageHDU (n_streaks, nwave) — propagated 1-σ errors |
| 4 | WAVE | ImageHDU (nwave,) — wavelength grid (Å) for SPECTRA/ERRORS |

The INFO table carries a global running index `streak_id`, the within-shot
streak number `i`, and per streak: an on-track spaxel centroid
(`ra_cen_spax`/`dec_cen_spax`), observed segment endpoints
(`ra_start`/`dec_start`, `ra_end`/`dec_end`), `seg_len_arcsec`, the streak PA
`streak_pa` (from the catalog track line), the track geometry
(`streak_slope`/`streak_intercept`), the shot MJD (`mjd_shot`) and exposure time
(`exptime`) from the ifu-index, and SDSS-g photometry. RA/Dec are stored as
float32. HET site coordinates are in the primary header
(SITELAT/SITELONG/SITEELEV).

## Imports

In [ ]:
import warnings
from astropy.utils.exceptions import AstropyWarning, AstropyUserWarning
from astropy.io.fits.verify import VerifyWarning
warnings.simplefilter('ignore', AstropyWarning)
warnings.simplefilter('ignore', AstropyUserWarning)
warnings.simplefilter('ignore', VerifyWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

import os.path as op
import numpy as np
from astropy.table import Table
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from joblib import Parallel, delayed
from tqdm import tqdm
import speclite.filters

## Configuration

In [ ]:
# ── paths ────────────────────────────────────────────────────────────────────
pdr_dir = '/home/jovyan/Hobby-Eberly-Public/HETDEX/pdr/pdr1/'
if not op.exists(pdr_dir):
    pdr_dir = '/home/jovyan/work/pdr1/'

OUT_PATH = 'HETDEX_PDR1_sats.fits'

# ── satellite tracks catalogue ────────────────────────────────────────────────
# Each row: shotid expnum slope intercept  (DEC = intercept + RA_deg * slope)
SAT_TRACKS_URL  = ('https://raw.githubusercontent.com/HETDEX/hetdex_api/'
                   'master/known_issues/hdr3/satellite_tracks.txt')
SAT_TRACKS_PATH = op.join('satellite_tracks.txt')  # local working directory

import urllib.request
if not op.exists(SAT_TRACKS_PATH):
    print('Downloading satellite_tracks.txt ...')
    urllib.request.urlretrieve(SAT_TRACKS_URL, SAT_TRACKS_PATH)

SAT_TAB = Table.read(
    SAT_TRACKS_PATH,
    format='ascii',
    names=['shotid', 'expnum', 'slope', 'intercept'],
)
SAT_TAB['shotid'] = SAT_TAB['shotid'].astype(np.int64)
print(f'Loaded {len(SAT_TAB)} satellite track rows for '
      f'{len(np.unique(SAT_TAB["shotid"]))} unique shotids')

STREAK_SIZE = 6.0 * u.arcsec   # half-width used to assign spaxels to tracks

# ── SAT mask bit ─────────────────────────────────────────────────────────────
SAT_BIT = None
try:
    from dexcube import MaskBits
    SAT_BIT = int(MaskBits.SAT)
except Exception:
    try:
        from dexcube.mask import SAT as SAT_BIT
        SAT_BIT = int(SAT_BIT)
    except Exception:
        SAT_BIT = 1024  # 0x400 — from 04-MaskingOptions.ipynb
        print(f'Using hardcoded SAT bit {SAT_BIT}')
print(f'SAT_BIT = {SAT_BIT}  (0x{SAT_BIT:03X})')

# ── instrument / survey constants ────────────────────────────────────────────
PIXEL_SCALE    = 0.5          # arcsec / pixel
SPAXEL_AREA    = PIXEL_SCALE ** 2   # arcsec^2 per spaxel = 0.25
CDELT3         = 2.0          # Angstrom per spectral bin (uniform across PDR1)

# ── exposure-dilution correction (#1) ────────────────────────────────────────
# A HETDEX shot co-adds N_EXP dithered exposures, but a satellite crosses in
# only ONE of them, so the datacube dilutes the streak flux by ~N_EXP.
# Multiplying the summed flux/errors by N_EXP recovers the in-exposure
# brightness. First-order: assumes equal-weight exposures and the streak in
# exactly one of them. The applied factor is documented in the primary header
# (EXPDILUT, N_EXP); no per-streak column is stored. Set False to store raw.
APPLY_EXP_DILUTION = True
N_EXP              = 3            # standard number of dithered exposures per shot
print(f'Exposure dilution: APPLY={APPLY_EXP_DILUTION}, N_EXP={N_EXP} '
      f'(brightens g_mag by {2.5*np.log10(N_EXP):.2f} mag)')

# ── meteor exclusion ─────────────────────────────────────────────────────────
# A handful of shots trip flag_satellite but are actually meteors on review.
# List their shotids here to drop them (applied when grouping into shots below).
METEOR_SHOTIDS = set()           # e.g. {20191008021, 20200114018}

# ── per-exposure timing (for ephemeris / archival satellite matching) ────────
# Recorded per streak as real quantities from the ifu-index: the shot MJD
# (mjd_shot), the affected dither number (expnum, from satellite_tracks.txt),
# and the exposure time (exptime). No dither cadence is assumed; reconstruct a
# per-dither start yourself if/when you have an authoritative cadence.

# ── HET site (topocentric observer for satellite ephemerides) ────────────────
SITELAT  =   30.681436   # deg N
SITELONG = -104.014744   # deg E (West negative)
SITEELEV =  2026.0       # m

# ── quality mask bits applied to satellite spaxels ───────────────────────────
# Channels/spaxels flagged with any of these bits are excluded from the sum.
# MAIN=1, FTF=2, BADPIX=8, BADAMP=16  (all others kept, including SAT itself)
BAD_BITS = 1 | 2 | 8 | 16   # = 27
print(f'Quality mask bits: BAD_BITS = {BAD_BITS} '
      f'(MAIN=1, FTF=2, BADPIX=8, BADAMP=16)')

# ── parallelism ──────────────────────────────────────────────────────────────
N_JOBS         = 8            # parallel workers; set to -1 for all cores
N_REVIEW       = None         # set to an integer to process only the first N shots

# ── speclite filter (loaded per-call inside _gmag for joblib compatibility) ──
_test_filt = speclite.filters.load_filters('sdss2010-g')
print(f'speclite filter ok: {_test_filt.names}')
del _test_filt

# ── shared wavelength array (uniform across PDR1) ────────────────────────────
_NWAVE_DEFAULT = 1036
WAVE_REF = 3470.0 + np.arange(_NWAVE_DEFAULT) * CDELT3  # Angstrom
print(f'Default wave array: {WAVE_REF[0]:.1f} - {WAVE_REF[-1]:.1f} Angstrom  ({len(WAVE_REF)} channels)')

## Load IFU index and group by shotid

In [ ]:
ifu_data = Table.read(op.join(pdr_dir, 'ifu-index.fits'))
print(f'IFU index: {len(ifu_data)} rows total')

sat_rows = ifu_data[ifu_data['flag_satellite'] < 0.9]
print(f'{len(sat_rows)} IFU rows flagged with significant satellite streaks')

# group into {shotid: [row, row, …]}
shot_groups = {}
for row in sat_rows:
    sid = int(row['shotid'])
    shot_groups.setdefault(sid, []).append(row)

# ── exclude shots that are actually meteors, not satellites ──────────────────
if METEOR_SHOTIDS:
    n_before = len(shot_groups)
    for sid in METEOR_SHOTIDS:
        shot_groups.pop(int(sid), None)
    print(f'Excluded {n_before - len(shot_groups)} meteor shot(s): '
          f'{sorted(int(s) for s in METEOR_SHOTIDS)}')

all_shotids = sorted(shot_groups.keys())
if N_REVIEW is not None:
    all_shotids = all_shotids[:N_REVIEW]
print(f'{len(all_shotids)} unique shotids to process')

## Helper functions

In [ ]:
def _wave_array(hdr):
    """Reconstruct wavelength array (Å) from cube header."""
    nwave = hdr['NAXIS3']
    crval = hdr.get('CRVAL3', 3470.0)
    cdelt = hdr.get('CDELT3', hdr.get('CD3_3', CDELT3))
    crpix = hdr.get('CRPIX3', 1.0)
    return crval + (np.arange(nwave) + 1 - crpix) * cdelt


def _sky_pa(hdr):
    """Sky PA of the +y pixel axis (deg E of N) from the 2-D WCS."""
    wcs2d = WCS(hdr).celestial
    cx, cy = wcs2d.wcs.crpix[0] - 1, wcs2d.wcs.crpix[1] - 1
    ra0, dec0 = wcs2d.wcs_pix2world(cx, cy,     0)
    ra1, dec1 = wcs2d.wcs_pix2world(cx, cy + 1, 0)
    dra  = (ra1 - ra0) * np.cos(np.deg2rad(dec0))
    ddec = dec1 - dec0
    return float(np.degrees(np.arctan2(dra, ddec)) % 360.0)


def _streak_pa_sky(wcs2d, xx, yy):
    """On-sky PA (deg E of N, in [0, 180)) of the streak, measured directly
    through the WCS so it is correct regardless of image parity (handedness).

    Fit the streak's principal axis in pixel space (SVD), map a short step along
    that axis through the WCS to sky coordinates, and take the position angle of
    the resulting (dE, dN). Replaces the earlier pixel-angle formula
    (pa_y + 90 - theta), which assumed a fixed handedness and reflected the PA
    on standard East-left cubes."""
    xx = np.asarray(xx, dtype=float)
    yy = np.asarray(yy, dtype=float)
    if len(xx) < 2:
        return np.nan
    pts = np.column_stack([xx - xx.mean(), yy - yy.mean()])
    _, _, vt = np.linalg.svd(pts, full_matrices=False)
    vx, vy = vt[0]
    x0, y0 = float(xx.mean()), float(yy.mean())
    (ra0, dec0), (ra1, dec1) = wcs2d.wcs_pix2world(
        [[x0, y0], [x0 + vx, y0 + vy]], 0)
    dE = (ra1 - ra0) * np.cos(np.deg2rad(dec0))
    dN = dec1 - dec0
    return float(np.degrees(np.arctan2(dE, dN)) % 180.0)


def _gmag(wave_aa, flux_density):
    """
    Compute SDSS-g AB magnitude using speclite.
    flux_density : 1-D array in erg/s/cm2/Angstrom (no units attached)
    wave_aa      : 1-D array in Angstrom (no units attached)
    Spectrum is scaled to 1e-17 units, padded to cover the filter,
    then passed to get_ab_magnitudes.
    Returns float AB magnitude, or NaN if the convolution fails.
    """
    if wave_aa is None or flux_density is None:
        return np.nan
    try:
        gfilt = speclite.filters.load_filters('sdss2010-g')
        # scale to 1e-17 erg/s/cm2/AA and attach units
        flux_scaled = np.array(flux_density) * u.Unit('1e-17 erg / (s cm2 AA)')
        wave_q      = np.array(wave_aa) * u.AA
        flux_pad, wave_pad = gfilt.pad_spectrum(flux_scaled, wave_q)
        mag_tbl = gfilt.get_ab_magnitudes(flux_pad, wave_pad)
        mag = float(mag_tbl['sdss2010-g'][0])
        return mag
    except Exception as e:
        print(f'_gmag failed: {type(e).__name__}: {e}')
        return np.nan


def extract_ifu(cube_path, sat_bit):
    """
    Open one IFU datacube and return per-spaxel data for SAT-masked pixels.

    Quality masking: channels flagged with MAIN(1), FTF(2), BADPIX(8), or
    BADAMP(16) are zeroed out in flux and variance. All other bits (including
    SAT itself) are ignored so satellite flux is preserved.
    n_spax counts only spaxels not flagged bad at the spatial level;
    this is used for the streak area and surface brightness calculation.

    Returns dict with keys:
        wave      : 1-D wavelength array (Angstrom)
        flux      : (nwave, n_spax_total) flux density in erg/s/cm2/Angstrom
        var       : (nwave, n_spax_total) variance
        n_spax    : int, SAT spaxels NOT flagged bad (used for area)
        streak_pa : on-sky PA of streak in this IFU (deg)
        pa_y      : sky PA of +y axis (deg)
        img       : (ny, nx) white-light image (3800-5200 AA)
        streak    : (ny, nx) bool SAT mask
        spax_ra   : (n_spax_total,) RA of each SAT spaxel centre (deg)
        spax_dec  : (n_spax_total,) Dec of each SAT spaxel centre (deg)
    or None if no SAT pixels found.
    """
    with fits.open(cube_path, memmap=True) as hdul:
        data = hdul['DATA'].data    # (nwave, ny, nx)
        err  = hdul['ERROR'].data
        mask = hdul['MASK'].data
        hdr  = hdul['DATA'].header

    wave   = _wave_array(hdr)
    pa_y   = _sky_pa(hdr)
    streak = (mask[0] & sat_bit) != 0

    if not streak.any():
        return None

    yy, xx = np.where(streak)

    # ── on-sky RA/Dec of each SAT spaxel (for track assignment) ──────────────
    wcs2d = WCS(hdr).celestial
    spax_ra, spax_dec = wcs2d.wcs_pix2world(xx.astype(float), yy.astype(float), 0)

    # ── extract spectra with quality masking ──────────────────────────────────
    spec = data[:, yy, xx]      # (nwave, n_spax)
    eps  = err[:, yy, xx]
    msk  = mask[:, yy, xx]      # full per-channel mask

    # Bits to mask: MAIN=1, FTF=2, BADPIX=8, BADAMP=16  (sum=27)
    # These indicate fundamentally bad data; all other flags (e.g. LARGEGAL,
    # METEOR, BADSHOT, BADCAL, PIXMASK, BADDET) are left in so we keep
    # the satellite flux even when other pipeline flags are present.
    BAD_BITS = 1 | 2 | 8 | 16   # = 27
    bad_channels = (msk & BAD_BITS) != 0
    good = np.isfinite(spec) & np.isfinite(eps) & ~bad_channels

    spec = np.where(good, spec, 0.0)
    var  = np.where(good, eps**2, 0.0)

    # convert from erg/s/cm2 per 2AA bin -> erg/s/cm2/AA flux density
    flux_density = spec / CDELT3
    var_density  = var  / CDELT3**2

    # n_spax for area: exclude spaxels that are bad at the spatial level
    # (bad in channel 0 as a proxy — these contribute no usable data)
    bad_spax = (mask[0, yy, xx] & BAD_BITS) != 0
    n_spax_good = int((~bad_spax).sum())

    # PA measured directly through the WCS (parity-correct); pa_y kept for reference
    pa_sky = _streak_pa_sky(wcs2d, xx.astype(float), yy.astype(float))

    # white-light image (3800-5200 AA) for review plot
    band = (wave >= 3800) & (wave <= 5200)
    cube_band = data[band].astype(np.float32)
    cube_band[~np.isfinite(cube_band)] = np.nan
    img = np.nanmean(cube_band, axis=0)

    return dict(
        wave=wave,
        flux=flux_density,
        var=var_density,
        n_spax=n_spax_good,
        streak_pa=pa_sky,
        pa_y=pa_y,
        img=img,
        streak=streak,
        spax_ra=spax_ra,
        spax_dec=spax_dec,
    )

## Per-shot processing function

Called in parallel — one call per shotid.
Returns a list of streak-level result dicts (one per distinct satellite pass).

In [ ]:
def _assign_spaxels_to_tracks(spax_ra, spax_dec, track_rows, streak_size=6.0):
    """
    Assign each SAT spaxel to a satellite track from satellite_tracks.txt.

    Track line: DEC = intercept + RA_deg * slope  (raw deg, no projection)
    Matches the hetdex_api convention exactly: a spaxel is on the track if
    its separation from the nearest point on the sampled line is < streak_size.
    Here we use the analytic perpendicular distance to the infinite line,
    which is equivalent and fully vectorised.

    The line  b*RA - DEC + a = 0  has perpendicular distance:
        d = |b*RA - DEC + a| / sqrt(b^2 + 1)
    in raw (RA_deg, DEC_deg) space.  streak_size is converted to degrees.
    No cos(dec) correction — this is how the catalogue was built.

    Returns
    -------
    labels : int array (n_spax,), track index 0..N-1, or -1 if unassigned
    """
    spax_ra  = np.asarray(spax_ra,  dtype=float)
    spax_dec = np.asarray(spax_dec, dtype=float)
    n_spax   = len(spax_ra)
    n_tracks = len(track_rows)

    streak_size_deg = streak_size / 3600.0  # arcsec -> deg

    # (n_tracks, n_spax) perpendicular distances in raw deg
    dist_deg = np.empty((n_tracks, n_spax), dtype=float)
    for k, row in enumerate(track_rows):
        b = float(row['slope'])
        a = float(row['intercept'])
        # line: b*RA - DEC + a = 0
        dist_deg[k] = np.abs(b * spax_ra - spax_dec + a) / np.sqrt(b**2 + 1.0)

    # assign each spaxel to its nearest track if within streak_size
    nearest  = np.argmin(dist_deg, axis=0)           # (n_spax,)
    min_dist = dist_deg[nearest, np.arange(n_spax)]  # (n_spax,)
    labels   = np.where(min_dist <= streak_size_deg, nearest, -1)
    return labels


def process_shot(shotid, ifu_rows, pdr_dir, sat_bit, sat_tab, streak_size_arcsec=6.0,
                 apply_dilution=True, n_exp=3):
    """
    Extract all satellite streak data for one shotid.

    Steps
    -----
    1. Look up satellite track lines for this shot from satellite_tracks.txt.
    2. Open every flagged IFU cube; extract SAT spaxels with quality masking.
    3. Assign each spaxel to a track by proximity in RA/Dec.
    4. For each track: sum flux/var, compute magnitudes, SB, S/N.

    Returns
    -------
    list of dicts (one per streak/track).
    """
    # ── get track lines for this shot ────────────────────────────────────────
    sel_shot  = sat_tab['shotid'] == shotid
    track_rows = list(sat_tab[sel_shot])  # list of rows; one per distinct satellite pass
    n_tracks   = len(track_rows)

    # shot MJD + exposure time (per-exposure timing model reference); from ifu-index
    try:
        shot_mjd = float(ifu_rows[0]['mjd'])
    except Exception:
        shot_mjd = np.nan
    try:
        exptime_val = float(ifu_rows[0]['exptime'])
    except Exception:
        exptime_val = np.nan

    ifu_results = []
    wave_ref    = None

    for row in ifu_rows:
        ifuslot   = str(row['ifuslot']).strip()
        cube_path = op.join(pdr_dir, 'datacubes', str(shotid),
                            f'dex_cube_{shotid}_{ifuslot}.fits')
        if not op.exists(cube_path):
            continue
        try:
            result = extract_ifu(cube_path, sat_bit)
        except Exception:
            continue
        if result is None:
            continue

        if wave_ref is None:
            wave_ref = result['wave']

        ifu_results.append(dict(
            ifuslot   = ifuslot,
            ra_cen    = float(row['ra_cen']),
            dec_cen   = float(row['dec_cen']),
            flag_sat  = float(row['flag_satellite']),
            flux      = result['flux'],
            var       = result['var'],
            n_spax    = result['n_spax'],
            streak_pa = result['streak_pa'],
            img       = result['img'],
            streak    = result['streak'],
            spax_ra   = result['spax_ra'],
            spax_dec  = result['spax_dec'],
        ))

    if not ifu_results:
        return []

    wave_for_mag = wave_ref if wave_ref is not None else WAVE_REF

    # ── assign spaxels to tracks ──────────────────────────────────────────────
    # If no track info available for this shot, fall back to putting everything
    # in streak 0 (should not happen for flagged shots, but be safe)
    if n_tracks == 0:
        track_rows_used = [None]  # sentinel: one unnamed streak
        for ifu in ifu_results:
            ifu['spax_track'] = np.zeros(ifu['n_spax'], dtype=int)
        n_tracks = 1
    else:
        track_rows_used = track_rows
        for ifu in ifu_results:
            ifu['spax_track'] = _assign_spaxels_to_tracks(
                ifu['spax_ra'], ifu['spax_dec'],
                track_rows, streak_size=streak_size_arcsec
            )

    # ── build one streak record per track ─────────────────────────────────────
    streak_records = []
    for streak_id in range(n_tracks):
        # collect the flux/var columns belonging to this track across all IFUs
        flux_chunks = []
        var_chunks  = []
        ifu_panels  = []
        ra_list, dec_list, w_list = [], [], []
        ra_chunks, dec_chunks = [], []
        pa_list = []

        for ifu in ifu_results:
            sel = ifu['spax_track'] == streak_id
            if not sel.any():
                continue
            flux_chunks.append(ifu['flux'][:, sel])  # (nwave, n_sel)
            var_chunks.append(ifu['var'][:, sel])
            ra_chunks.append(np.asarray(ifu['spax_ra'])[sel])
            dec_chunks.append(np.asarray(ifu['spax_dec'])[sel])
            ra_list.append(ifu['ra_cen'])
            dec_list.append(ifu['dec_cen'])
            w_list.append(int(sel.sum()))
            if np.isfinite(ifu['streak_pa']):
                pa_list.append(ifu['streak_pa'])
            # build per-IFU streak mask restricted to this track's spaxels
            # (used for the contour in review plots)
            streak_for_track = ifu['streak'].copy()
            yy, xx = np.where(ifu['streak'])
            for k_spax, keep in enumerate(sel):
                if not keep:
                    streak_for_track[yy[k_spax], xx[k_spax]] = False
            ifu_panels.append(dict(
                ifuslot   = ifu['ifuslot'],
                n_spax    = int(sel.sum()),
                img       = ifu['img'],
                streak    = streak_for_track,
            ))

        if not flux_chunks:
            continue  # no spaxels assigned to this track

        all_flux = np.concatenate(flux_chunks, axis=1)  # (nwave, N)
        all_var  = np.concatenate(var_chunks,  axis=1)
        summed_flux  = all_flux.sum(axis=1).astype(np.float32)
        summed_err   = np.sqrt(all_var.sum(axis=1)).astype(np.float32)

        # ── exposure-dilution correction (#1) ────────────────────────────────
        # The streak is present in one of n_exp co-added exposures; scale up to
        # recover the in-exposure brightness. Stored spectra are corrected;
        # divide by 'exp_dilution' for raw cube-averaged flux. 1.0 if disabled.
        dilution     = float(n_exp) if apply_dilution else 1.0
        summed_flux  = (summed_flux * dilution).astype(np.float32)
        summed_err   = (summed_err  * dilution).astype(np.float32)

        n_spax_total = int(sum(w_list))
        n_ifu        = len(flux_chunks)

        area_arcsec2 = n_spax_total * SPAXEL_AREA
        median_pa = float(np.median(pa_list)) if pa_list else np.nan

        g_mag = _gmag(wave_for_mag, summed_flux)
        sb_mag_arcsec2 = float(g_mag + 2.5 * np.log10(area_arcsec2)) \
            if (np.isfinite(g_mag) and area_arcsec2 > 0) else np.nan

        gband_mask = (wave_for_mag >= 3800) & (wave_for_mag <= 5500)
        with np.errstate(invalid='ignore', divide='ignore'):
            snr_arr = np.where(summed_err[gband_mask] > 0,
                               summed_flux[gband_mask] / summed_err[gband_mask], np.nan)
        mean_snr = float(np.nanmean(snr_arr)) if np.any(np.isfinite(snr_arr)) else np.nan

        # store track slope/intercept for reference
        tr = track_rows_used[streak_id] if track_rows_used[streak_id] is not None else None
        slope_val     = float(tr['slope'])     if tr is not None else np.nan
        intercept_val = float(tr['intercept']) if tr is not None else np.nan
        expnum_val    = int(tr['expnum'])      if tr is not None else -1

        # ── flux-independent on-track geometry ───────────────────────────────
        ra_arr  = np.concatenate(ra_chunks)
        dec_arr = np.concatenate(dec_chunks)
        # unwrap RA near the first spaxel to avoid the 0h/360 seam, then centroid
        ra_ref  = float(ra_arr[0])
        ra_uw   = ((ra_arr - ra_ref + 180.0) % 360.0) - 180.0 + ra_ref
        ra_cen_uw    = float(np.mean(ra_uw))
        dec_cen_spax = float(np.mean(dec_arr))
        ra_cen_spax  = ra_cen_uw % 360.0
        cosd = float(np.cos(np.deg2rad(dec_cen_spax)))
        dE = (ra_uw - ra_cen_uw) * cosd * 3600.0   # arcsec East of centroid
        dN = (dec_arr - dec_cen_spax) * 3600.0     # arcsec North of centroid

        # PA of the satellite track (deg E of N); catalog slope when available
        if np.isfinite(slope_val):
            streak_pa_track = float(np.degrees(np.arctan2(cosd, slope_val)) % 180.0)
        else:
            _u, _s, _vt = np.linalg.svd(np.column_stack([dE, dN]), full_matrices=False)
            streak_pa_track = float(np.degrees(np.arctan2(_vt[0][0], _vt[0][1])) % 180.0)

        # project spaxels along the track direction; extrema -> observed endpoints
        uE = np.sin(np.deg2rad(streak_pa_track)); uN = np.cos(np.deg2rad(streak_pa_track))
        proj = dE * uE + dN * uN
        tmin = float(proj.min()); tmax = float(proj.max())
        seg_len_arcsec = tmax - tmin
        ra_beg  = (ra_cen_uw + (tmin * uE / 3600.0) / cosd) % 360.0
        dec_beg =  dec_cen_spax + (tmin * uN / 3600.0)
        ra_end  = (ra_cen_uw + (tmax * uE / 3600.0) / cosd) % 360.0
        dec_end =  dec_cen_spax + (tmax * uN / 3600.0)

        # shot MJD recorded directly (no cadence model; see config note)
        mjd_shot = shot_mjd

        streak_records.append(dict(
            shotid         = shotid,
            streak_id      = streak_id,
            n_ifu          = n_ifu,
            n_spax         = n_spax_total,
            area_arcsec2   = area_arcsec2,
            ra_cen_spax    = ra_cen_spax,
            dec_cen_spax   = dec_cen_spax,
            ra_start       = ra_beg,
            dec_start      = dec_beg,
            ra_end         = ra_end,
            dec_end        = dec_end,
            spax_ra        = ra_arr.astype(np.float32),
            spax_dec       = dec_arr.astype(np.float32),
            seg_len_arcsec = seg_len_arcsec,
            streak_pa_track = streak_pa_track,
            slope          = slope_val,
            intercept      = intercept_val,
            exptime        = exptime_val,
            mjd_shot       = mjd_shot,
            g_mag          = g_mag,
            sb_mag_arcsec2 = sb_mag_arcsec2,
            mean_snr       = mean_snr,
            summed_flux    = summed_flux,
            summed_err     = summed_err,
            ifu_panels     = ifu_panels,
        ))

    return streak_records

## Parallel extraction over all shotids

In [ ]:
raw_results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=0)(
    delayed(process_shot)(
        shotid, shot_groups[shotid], pdr_dir, SAT_BIT, SAT_TAB,
        streak_size_arcsec=6.0, apply_dilution=APPLY_EXP_DILUTION, n_exp=N_EXP
    )
    for shotid in tqdm(all_shotids, desc='shots')
)

# flatten list-of-lists; each element is one streak record
all_streaks = [rec for shot_list in raw_results for rec in shot_list]

n_shots_with_data = sum(1 for s in raw_results if len(s) > 0)
print(f'\nProcessed {len(all_shotids)} shots')
print(f'  {n_shots_with_data} shots yielded at least one streak')
print(f'  {len(all_streaks)} total streak records')
multi = sum(1 for s in raw_results if len(s) > 1)
print(f'  {multi} shots with ≥2 distinct satellite passes')

In [ ]:
print(len(all_streaks), 'streaks;', n_shots_with_data, 'shots')   # expect ~533 / ~497

## Save FITS output

Structure:
- `PRIMARY`  — run metadata  
- `INFO`     — BinTableHDU, one row per streak  
- `SPECTRA`  — ImageHDU `(n_streaks, nwave)`, summed flux in erg/s/cm²/Å  
- `ERRORS`   — ImageHDU `(n_streaks, nwave)`, propagated 1-σ errors
- `WAVE`     — ImageHDU `(nwave,)`, wavelength grid in Å (axis 1 of SPECTRA/ERRORS)

In [ ]:
if not all_streaks:
    print('No streaks found — nothing to save.')
else:
    n_str  = len(all_streaks)
    nwave  = len(all_streaks[0]['summed_flux'])
    # wavelength solution for SPECTRA/ERRORS axis 1 (matches WAVE_REF)
    wave_out = (3470.0 + np.arange(nwave) * CDELT3).astype(np.float64)  # Angstrom

    # ── PRIMARY ──────────────────────────────────────────────────────────────
    primary = fits.PrimaryHDU()
    primary.header['NSTREAKS'] = (n_str,      'total number of streak records')
    primary.header['NSHOTS']   = (n_shots_with_data, 'shots with >=1 streak')
    primary.header['SAT_BIT']  = (SAT_BIT,    'SAT mask bit value')
    primary.header['PIX_SCAL'] = (PIXEL_SCALE, 'arcsec per pixel')
    primary.header['CDELT3']   = (CDELT3,      'Angstrom per spectral bin')
    primary.header['FILTER']   = ('sdss2010-g', 'filter used for g-band photometry')
    primary.header['EXPDILUT'] = (APPLY_EXP_DILUTION, 'exposure-dilution correction applied')
    primary.header['N_EXP']    = (N_EXP,      'exposures co-added per shot (dilution factor)')
    primary.header['CRVAL1W']  = (float(wave_out[0]), 'start wavelength of SPECTRA axis (Angstrom)')
    primary.header['SITELAT']  = (30.681436,   'HET site latitude (deg N)')
    primary.header['SITELONG'] = (-104.014744, 'HET site longitude (deg E, West negative)')
    primary.header['SITEELEV'] = (2026.0,      'HET site elevation (m)')
    primary.header['COMMENT']  = 'HETDEX PDR1 satellite streak spectra - per shotid'

    # ── INFO table ────────────────────────────────────────────────────────────
    index_tbl = Table({
        'streak_id':      np.arange(n_str, dtype=np.int32),
        'shotid':         np.array([r['shotid']         for r in all_streaks], dtype=np.int64),
        'i':              np.array([r['streak_id']      for r in all_streaks], dtype=np.int16),
        'n_ifu':          np.array([r['n_ifu']          for r in all_streaks], dtype=np.int16),
        'n_spax':         np.array([r['n_spax']         for r in all_streaks], dtype=np.int32),
        'area_arcsec2':   np.array([r['area_arcsec2']   for r in all_streaks], dtype=np.float32),
        'ra_cen_spax':    np.array([r['ra_cen_spax']    for r in all_streaks], dtype=np.float32),
        'dec_cen_spax':   np.array([r['dec_cen_spax']   for r in all_streaks], dtype=np.float32),
        'ra_start':       np.array([r['ra_start']       for r in all_streaks], dtype=np.float32),
        'dec_start':      np.array([r['dec_start']      for r in all_streaks], dtype=np.float32),
        'ra_end':         np.array([r['ra_end']         for r in all_streaks], dtype=np.float32),
        'dec_end':        np.array([r['dec_end']        for r in all_streaks], dtype=np.float32),
        'seg_len_arcsec': np.array([r['seg_len_arcsec'] for r in all_streaks], dtype=np.float32),
        'streak_pa':      np.array([r['streak_pa_track'] for r in all_streaks], dtype=np.float32),
        'streak_slope':   np.array([r['slope']          for r in all_streaks], dtype=np.float32),
        'streak_intercept': np.array([r['intercept']    for r in all_streaks], dtype=np.float32),
        'exptime':        np.array([r['exptime']        for r in all_streaks], dtype=np.float32),
        'mjd_shot':       np.array([r['mjd_shot']       for r in all_streaks], dtype=np.float64),
        'g_mag':          np.array([r['g_mag']          for r in all_streaks], dtype=np.float32),
        'sb_mag_arcsec2': np.array([r['sb_mag_arcsec2'] for r in all_streaks], dtype=np.float32),
        'mean_snr':       np.array([r['mean_snr']       for r in all_streaks], dtype=np.float32),
    })

    # column descriptions
    index_tbl['streak_id'     ].description = 'global running index of streak record (0-based, unique)'
    index_tbl['shotid'        ].description = 'HETDEX shot identifier'
    index_tbl['i'             ].description = 'streak index within shot (0-based)'
    index_tbl['n_ifu'         ].description = 'number of IFUs contributing to streak'
    index_tbl['n_spax'        ].description = 'total SAT spaxels summed'
    index_tbl['area_arcsec2'  ].description = 'streak area (n_spax × 0.25 arcsec²)'
    index_tbl['ra_cen_spax'   ].description = 'flux-independent centroid RA of SAT spaxels, on the streak (deg)'
    index_tbl['dec_cen_spax'  ].description = 'flux-independent centroid Dec of SAT spaxels, on the streak (deg)'
    index_tbl['ra_start'      ].description = 'RA of observed streak endpoint at min track projection (deg)'
    index_tbl['dec_start'     ].description = 'Dec of observed streak endpoint at min track projection (deg)'
    index_tbl['ra_end'        ].description = 'RA of observed streak endpoint at max track projection (deg)'
    index_tbl['dec_end'       ].description = 'Dec of observed streak endpoint at max track projection (deg)'
    index_tbl['seg_len_arcsec'].description = 'length of observed streak segment along the track (arcsec)'
    index_tbl['streak_pa'     ].description = 'PA of satellite track (from streak_slope), deg E of N [0,180)'
    index_tbl['streak_slope'  ].description = 'satellite track slope from satellite_tracks.txt (DEC=intercept+RA*slope)'
    index_tbl['streak_intercept'].description = 'satellite track intercept from satellite_tracks.txt (deg)'
    index_tbl['exptime'       ].description = 'exposure time from ifu-index (s)'
    index_tbl['mjd_shot'      ].description = 'shot MJD from ifu-index (UTC)'
    index_tbl['g_mag'         ].description = 'SDSS-g AB magnitude of summed streak spectrum'
    index_tbl['sb_mag_arcsec2'].description = 'surface brightness g_mag + 2.5*log10(area) mag/arcsec²'
    index_tbl['mean_snr'      ].description = 'mean S/N per pixel in g-band window (3800–5500 Å)'

    index_hdu = fits.BinTableHDU(index_tbl, name='INFO')

    # ── SPECTRA and ERRORS image HDUs ─────────────────────────────────────────
    spectra_arr = np.vstack([r['summed_flux'] for r in all_streaks]).astype(np.float32)  # (n_str, nwave)
    errors_arr  = np.vstack([r['summed_err']  for r in all_streaks]).astype(np.float32)

    spectra_hdu = fits.ImageHDU(spectra_arr, name='SPECTRA')
    spectra_hdu.header['CTYPE1'] = ('WAVE',     'wavelength axis')
    spectra_hdu.header['CRPIX1'] = (1.0,        'reference pixel (1-based)')
    spectra_hdu.header['CRVAL1'] = (float(wave_out[0]), 'wavelength at reference pixel (Angstrom)')
    spectra_hdu.header['CDELT1'] = (float(CDELT3),      'wavelength increment (Angstrom)')
    spectra_hdu.header['CUNIT1'] = ('Angstrom', 'wavelength unit')
    spectra_hdu.header['BUNIT']   = 'erg/s/cm2/Angstrom'
    spectra_hdu.header['COMMENT'] = 'Row i corresponds to INFO row i. Shape: (n_streaks, nwave)'

    errors_hdu = fits.ImageHDU(errors_arr, name='ERRORS')
    errors_hdu.header['BUNIT']   = 'erg/s/cm2/Angstrom'
    errors_hdu.header['COMMENT'] = 'Propagated 1-sigma errors. Row i = INFO row i.'
    for _k, _v, _c in [('CTYPE1', 'WAVE', 'wavelength axis'),
                       ('CRPIX1', 1.0, 'reference pixel (1-based)'),
                       ('CRVAL1', float(wave_out[0]), 'wavelength at reference pixel (Angstrom)'),
                       ('CDELT1', float(CDELT3), 'wavelength increment (Angstrom)'),
                       ('CUNIT1', 'Angstrom', 'wavelength unit')]:
        errors_hdu.header[_k] = (_v, _c)

    # ── WAVE HDU (#5): explicit wavelength array for SPECTRA/ERRORS ───────────
    wave_hdu = fits.ImageHDU(wave_out, name='WAVE')
    wave_hdu.header['BUNIT']   = 'Angstrom'
    wave_hdu.header['COMMENT'] = 'Wavelength grid for SPECTRA/ERRORS columns (axis 1).'

    # ── write ─────────────────────────────────────────────────────────────────
    fits.HDUList([primary, index_hdu, spectra_hdu, errors_hdu, wave_hdu]).writeto(
        OUT_PATH, overwrite=True
    )
    print(f'Wrote {OUT_PATH}')
    print(f'  INFO     : {n_str} rows × {len(index_tbl.colnames)} columns')
    print(f'  SPECTRA  : {spectra_arr.shape}')
    print(f'  ERRORS   : {errors_arr.shape}')

## Sanity-check summary

In [ ]:
if all_streaks:
    with fits.open(OUT_PATH) as hdul:
        hdul.info()

    t = Table.read(OUT_PATH, hdu='INFO')
    print()
    print('INFO table preview:')
    t['shotid', 'streak_id', 'n_ifu', 'n_spax', 'area_arcsec2',
      'g_mag', 'sb_mag_arcsec2', 'mean_snr', 'streak_pa'][:10].pprint(max_width=120)

    print()
    print('g-band magnitude statistics:')
    gmags = t['g_mag'][np.isfinite(t['g_mag'])]
    print(f'  N finite     : {len(gmags)}')
    print(f'  median g_mag : {np.median(gmags):.2f}')
    print(f'  range        : {gmags.min():.2f} – {gmags.max():.2f}')

    print()
    print('Surface brightness statistics (mag/arcsec²):')
    sbs = t['sb_mag_arcsec2'][np.isfinite(t['sb_mag_arcsec2'])]
    print(f'  N finite     : {len(sbs)}')
    print(f'  median SB    : {np.median(sbs):.2f}')
    print(f'  range        : {sbs.min():.2f} – {sbs.max():.2f}')

## Review plots: all shotids

One figure per shotid. IFU white-light images (top row/s) + summed spectra (bottom row).
Multiple streaks per shot get distinct colours. >4 IFUs triggers a 3-row layout.
Saved to `shot_review_plots/shot_{shotid}.png`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os

# ── tuneable layout constants ─────────────────────────────────────────────────
IMG_W        = 2.0   # inches per IFU image panel
IMG_H        = 2.0   # inches per image row
SPEC_H       = 2.4   # inches for spectrum row
IMGS_PER_ROW = 4     # max IFU images per image row (>4 triggers 3-row layout)
PLOT_DIR     = 'shot_review_plots'  # output directory

# colour cycle for streaks: index 0 -> C1 (red), 1 -> C2 (orange), ...
STREAK_COLORS = ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9']

os.makedirs(PLOT_DIR, exist_ok=True)

if all_streaks:
    # ── get shared wavelength array ───────────────────────────────────────────
    wave_plot = None
    for sid in all_shotids:
        for row in shot_groups[sid]:
            ifuslot   = str(row['ifuslot']).strip()
            cube_path = op.join(pdr_dir, 'datacubes', str(sid),
                                f'dex_cube_{sid}_{ifuslot}.fits')
            if op.exists(cube_path):
                with fits.open(cube_path, memmap=True) as hdul:
                    wave_plot = _wave_array(hdul['DATA'].header)
                break
        if wave_plot is not None:
            break

    t = Table.read(OUT_PATH, hdu='INFO')
    with fits.open(OUT_PATH) as hdul:
        spectra = hdul['SPECTRA'].data
        errors  = hdul['ERRORS'].data

    # ── build {shotid: [streak_record, ...]} map using in-memory all_streaks ──
    # all_streaks is already ordered; group consecutive records by shotid
    shot_streak_map = {}
    for idx, rec in enumerate(all_streaks):
        shot_streak_map.setdefault(rec['shotid'], []).append(
            dict(rec=rec, idx=idx, row=t[idx])
        )

    n_saved = 0
    for shotid, entries in shot_streak_map.items():

        # ── collect all IFU panels across all streaks in this shot ────────────
        # each panel carries its streak_id so we can colour the contour
        all_panels = []   # list of {ifuslot, n_spax, img, streak, streak_id}
        for e in entries:
            sid = e['rec']['streak_id']
            for p in e['rec']['ifu_panels']:
                all_panels.append(dict(
                    ifuslot   = p['ifuslot'],
                    n_spax    = p['n_spax'],
                    img       = p['img'],
                    streak    = p['streak'],
                    streak_id = sid,
                ))

        n_ifu     = len(all_panels)
        n_streaks = len(entries)

        # ── decide layout ─────────────────────────────────────────────────────
        # n_img_rows: 1 if <=4 IFUs, 2 if >4 (up to 8; wrap at IMGS_PER_ROW)
        n_img_rows  = int(np.ceil(n_ifu / IMGS_PER_ROW))
        n_total_rows = n_img_rows + 1   # +1 for spectrum row at bottom

        # columns = widest of: IFUs-per-image-row vs n_streaks (for spectra)
        n_cols   = max(min(n_ifu, IMGS_PER_ROW), n_streaks)
        fig_w    = n_cols * IMG_W
        fig_h    = n_img_rows * IMG_H + SPEC_H

        fig = plt.figure(figsize=(fig_w, fig_h))
        fig.suptitle(f'shotid {shotid}   ({n_streaks} streak(s), {n_ifu} IFU(s))',
                     fontsize=9, y=1.01)

        height_ratios = [IMG_H] * n_img_rows + [SPEC_H]
        gs = GridSpec(
            n_total_rows, n_cols,
            height_ratios=height_ratios,
            hspace=0.45,
            wspace=0.08,
        )

        # ── IFU image panels ──────────────────────────────────────────────────
        for j, p in enumerate(all_panels):
            img_row = j // IMGS_PER_ROW
            img_col = j  % IMGS_PER_ROW
            ax = fig.add_subplot(gs[img_row, img_col])

            img    = p['img']
            streak = p['streak']
            color  = STREAK_COLORS[p['streak_id'] % len(STREAK_COLORS)]

            finite = img[np.isfinite(img)]
            vmin, vmax = (np.nanpercentile(finite, [5, 99])
                          if finite.size else (0, 1))
            ax.imshow(img, origin='lower', cmap='gray',
                      vmin=vmin, vmax=vmax, interpolation='nearest')
            ax.contour(streak.astype(float), levels=[0.5],
                       colors=color, linewidths=0.9)
            ax.set_title(f"{p['ifuslot']}  {p['n_spax']}spx",
                         fontsize=6.5, pad=2, color=color)
            ax.set_xticks([])
            ax.set_yticks([])

        # hide any empty image slots in the last image row
        for j in range(n_ifu, n_img_rows * IMGS_PER_ROW):
            img_row = j // IMGS_PER_ROW
            img_col = j  % IMGS_PER_ROW
            if img_row < n_img_rows and img_col < n_cols:
                fig.add_subplot(gs[img_row, img_col]).set_visible(False)

        # ── spectrum panels (one per streak, bottom row) ───────────────────────
        # each spectrum spans n_cols/n_streaks columns so they fill the row evenly
        spec_row = n_img_rows
        cols_per_spec = max(1, n_cols // n_streaks)

        for k, e in enumerate(entries):
            col_start = k * cols_per_spec
            col_end   = col_start + cols_per_spec if k < n_streaks - 1 else n_cols
            ax_sp = fig.add_subplot(gs[spec_row, col_start:col_end])

            idx   = e['idx']
            row   = e['row']
            color = STREAK_COLORS[row['i'] % len(STREAK_COLORS)]
            flux  = spectra[idx]
            err   = errors[idx]

            ax_sp.plot(wave_plot, flux, lw=0.6, color=color)
            ax_sp.fill_between(wave_plot, flux - err, flux + err,
                               color=color, alpha=0.2, lw=0)
            ax_sp.axhline(0, color='k', lw=0.4, ls='--')
            ax_sp.set_xlim(wave_plot.min(), wave_plot.max())

            g_str  = f"{row['g_mag']:.2f}"           if np.isfinite(row['g_mag'])          else 'nan'
            sb_str = f"{row['sb_mag_arcsec2']:.2f}"  if np.isfinite(row['sb_mag_arcsec2']) else 'nan'
            pa_str = f"{row['streak_pa']:.0f}"   if np.isfinite(row['streak_pa'])  else 'nan'
            ax_sp.set_title(
                f"streak {row['i']}  "
                f"n_ifu={row['n_ifu']}  n_spax={row['n_spax']}\n"
                f"g={g_str}  SB={sb_str} mag/arcsec2  PA={pa_str} deg",
                fontsize=6.5, pad=2, color=color
            )
            ax_sp.set_xlabel('Wavelength (Angstrom)', fontsize=7)
            if k == 0:
                ax_sp.set_ylabel('flux (1e-17 erg/s/cm2/AA)', fontsize=7)
            ax_sp.tick_params(labelsize=6.5)

        fname = op.join(PLOT_DIR, f'shot_{shotid}.png')
        fig.savefig(fname, dpi=130, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        n_saved += 1

    print(f'Saved {n_saved} plots to {PLOT_DIR}/')

## Focal-plane diagnostic (parallel)

For every shot, plot all IFU footprints, the SAT streak path (spaxels in RA/Dec),
and the recorded coordinate locations (`ra_start`/`dec_start`, `ra_end`/`dec_end`,
and the `ra_cen_spax`/`dec_cen_spax` centroid) to verify the endpoints land on the
streak within the focal plane. One figure per shot in `focal_plane_diagnostics/`,
generated in parallel over shots.

In [ ]:
import os

def _diagnose(payload):
    import os.path as op
    import numpy as np
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    import matplotlib.transforms as mtransforms

    IFU_SIZE = 51.0
    COLORS   = ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9']
    FP_DIR   = 'focal_plane_diagnostics'

    sid  = payload['shotid']
    ra0, dec0, field_pa = payload['ra0'], payload['dec0'], payload['field_pa']
    cosd = np.cos(np.deg2rad(dec0))

    def T(ra, dec):
        return ((np.asarray(ra, float) - ra0) * cosd * 3600.0,
                (np.asarray(dec, float) - dec0) * 3600.0)

    fig, ax = plt.subplots(figsize=(7.5, 7.5))
    # all IFU footprints (51" boxes, rotated by field PA)
    for cra, cdec in zip(payload['ifu_ra'], payload['ifu_dec']):
        cE, cN = T(cra, cdec)
        rect = Rectangle((cE - IFU_SIZE / 2, cN - IFU_SIZE / 2), IFU_SIZE, IFU_SIZE,
                         fill=False, ec='0.75', lw=0.5, zorder=1)
        rect.set_transform(
            mtransforms.Affine2D().rotate_deg_around(cE, cN, -field_pa) + ax.transData)
        ax.add_patch(rect)

    for k, s in enumerate(payload['streaks']):
        col = COLORS[k % len(COLORS)]
        sE, sN = T(s['spax_ra'], s['spax_dec'])
        ax.scatter(sE, sN, s=3, color=col, alpha=0.35, lw=0, zorder=2,
                   label=f"streak {s['streak_id']} spaxels")
        bE, bN = T(s['ra_start'], s['dec_start'])
        eE, eN = T(s['ra_end'],   s['dec_end'])
        gE, gN = T(s['ra_cen'],   s['dec_cen'])
        ax.plot([bE, eE], [bN, eN], '-', color=col, lw=1.0, zorder=4)
        ax.plot(bE, bN, 'o', mfc='lime', mec='k', ms=10, zorder=5, label='start' if k == 0 else None)
        ax.plot(eE, eN, 's', mfc='red',  mec='k', ms=10, zorder=5, label='end'   if k == 0 else None)
        ax.plot(gE, gN, 'X', mfc='cyan', mec='k', ms=11, zorder=5, label='centroid' if k == 0 else None)

    ax.set_aspect('equal')
    ax.invert_xaxis()   # East (increasing RA) to the left
    ax.set_xlabel(r'$\Delta$RA$\cdot\cos\delta$  (arcsec, E $\rightarrow$)')
    ax.set_ylabel(r'$\Delta$Dec  (arcsec, N $\uparrow$)')
    ax.set_title(f'shot {sid}   focal-plane streak diagnostic   '
                 f'({len(payload["streaks"])} streak(s))', fontsize=10)
    ax.legend(loc='upper right', fontsize=7, framealpha=0.9)
    fig.savefig(op.join(FP_DIR, f'focalplane_{sid}.png'), dpi=130, bbox_inches='tight')
    plt.close(fig)
    return sid


if all_streaks:
    os.makedirs('focal_plane_diagnostics', exist_ok=True)

    # group streak records by shot
    _shot_recs = {}
    for rec in all_streaks:
        _shot_recs.setdefault(int(rec['shotid']), []).append(rec)

    # lightweight payloads (do not ship the review-plot images to workers)
    _payloads = []
    for sid, recs in _shot_recs.items():
        ifus = ifu_data[ifu_data['shotid'] == sid]
        ra0  = float(np.mean(ifus['ra_cen']))
        dec0 = float(np.mean(ifus['dec_cen']))
        fpa  = float(np.nanmedian(np.asarray(ifus['pa'], float))) if 'pa' in ifus.colnames else 0.0
        streaks = [dict(streak_id=int(r['streak_id']),
                        spax_ra=np.asarray(r['spax_ra'],  float),
                        spax_dec=np.asarray(r['spax_dec'], float),
                        ra_start=float(r['ra_start']), dec_start=float(r['dec_start']),
                        ra_end=float(r['ra_end']),     dec_end=float(r['dec_end']),
                        ra_cen=float(r['ra_cen_spax']), dec_cen=float(r['dec_cen_spax']))
                   for r in recs]
        _payloads.append(dict(shotid=sid, ra0=ra0, dec0=dec0, field_pa=fpa,
                              ifu_ra=np.asarray(ifus['ra_cen'],  float),
                              ifu_dec=np.asarray(ifus['dec_cen'], float),
                              streaks=streaks))

    _done = Parallel(n_jobs=N_JOBS, backend='loky', verbose=0)(
        delayed(_diagnose)(p) for p in tqdm(_payloads, desc='focal-plane diag'))
    print(f'Saved {len(_done)} focal-plane diagnostics to focal_plane_diagnostics/')


In [ ]:
"""
Diagnostic cells to paste into the notebook to investigate
weird SAT mask patterns in shots from 2023-09 to 2024-04.
"""

# ── CELL 1: Look at the raw mask for a specific suspect shot ─────────────────
# Paste this into a notebook cell and set DIAG_SHOTID to one of the weird shots

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import os.path as op

DIAG_SHOTID = 20240314026   # change to any suspect shotid
SAT_BIT_CHECK = 1024

diag_rows = shot_groups[DIAG_SHOTID]
print(f"Shot {DIAG_SHOTID}: {len(diag_rows)} flagged IFU(s)")
print(f"flag_satellite values: {[float(r['flag_satellite']) for r in diag_rows]}")

for row in diag_rows:
    ifuslot   = str(row['ifuslot']).strip()
    cube_path = op.join(pdr_dir, 'datacubes', str(DIAG_SHOTID),
                        f'dex_cube_{DIAG_SHOTID}_{ifuslot}.fits')
    if not op.exists(cube_path):
        print(f"  {ifuslot}: cube missing")
        continue

    with fits.open(cube_path, memmap=True) as hdul:
        data  = hdul['DATA'].data    # (nwave, ny, nx)
        mask  = hdul['MASK'].data    # (nwave, ny, nx)
        hdr   = hdul['DATA'].header

    mask0     = mask[0]              # spatial mask slice
    sat_pix   = (mask0 & SAT_BIT_CHECK) != 0
    n_sat     = sat_pix.sum()

    # what OTHER bits are set on SAT pixels?
    other_bits_on_sat = mask0[sat_pix] & ~SAT_BIT_CHECK
    unique_other = np.unique(other_bits_on_sat)

    # how does the SAT bit vary with wavelength channel?
    sat_per_channel = ((mask & SAT_BIT_CHECK) != 0).sum(axis=(1, 2))

    print(f"\n  IFU {ifuslot}:")
    print(f"    SAT spaxels in channel 0 : {n_sat}")
    print(f"    Other bits set on SAT pix: {unique_other}  (0 = clean, else = also flagged)")
    print(f"    SAT spaxel count range across wavelength channels: "
          f"{sat_per_channel.min()} – {sat_per_channel.max()}")
    print(f"    Is SAT count constant across channels? {np.all(sat_per_channel == sat_per_channel[0])}")

    # ── plot 1: spatial distribution of SAT pixels ───────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    # white-light image
    band = np.nanmean(data[200:800].astype(float), axis=0)
    finite = band[np.isfinite(band)]
    vmin, vmax = np.nanpercentile(finite, [1, 99]) if finite.size else (0, 1)
    axes[0].imshow(band, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    axes[0].contour(sat_pix.astype(float), levels=[0.5], colors='red', linewidths=0.8)
    axes[0].set_title(f'{ifuslot}: white-light + SAT contour\n({n_sat} SAT spaxels)')
    axes[0].set_xticks([]); axes[0].set_yticks([])

    # SAT mask itself
    axes[1].imshow(sat_pix.astype(float), origin='lower', cmap='Reds')
    axes[1].set_title('SAT bit mask (channel 0)')
    axes[1].set_xticks([]); axes[1].set_yticks([])

    # SAT count vs wavelength channel — tells us if it's a 2D or 3D mask
    axes[2].plot(sat_per_channel, lw=0.8, color='C1')
    axes[2].set_xlabel('Wavelength channel index')
    axes[2].set_ylabel('N pixels with SAT bit set')
    axes[2].set_title('SAT pixel count vs wavelength\n(flat = 2D mask, varying = 3D mask)')

    plt.suptitle(f'Shot {DIAG_SHOTID}  IFU {ifuslot}', fontsize=10)
    plt.tight_layout()
    plt.show()

    # ── plot 2: all unique mask bit combinations on SAT pixels ───────────────
    print(f"\n    Unique full mask values at SAT spaxels (channel 0):")
    unique_full, counts = np.unique(mask0[sat_pix], return_counts=True)
    for val, cnt in sorted(zip(counts, unique_full), reverse=True):
        bits = [b for b in range(16) if val & (1 << b)]
        print(f"      mask=0b{unique_full[counts==cnt][0]:016b}  (bits {bits})  n={cnt}")


# ── CELL 2: Compare a good shot vs a bad shot mask morphology ────────────────
# Run this to see if the scattered-circle pattern is date-dependent

GOOD_SHOT = 20240501010   # known good (clean diagonal streak)
BAD_SHOT  = 20240314026   # known bad  (scattered circles)

def quick_sat_image(shotid, shot_groups, pdr_dir, sat_bit=1024):
    rows = shot_groups.get(shotid, [])
    results = []
    for row in rows:
        ifuslot   = str(row['ifuslot']).strip()
        cube_path = op.join(pdr_dir, 'datacubes', str(shotid),
                            f'dex_cube_{shotid}_{ifuslot}.fits')
        if not op.exists(cube_path):
            continue
        with fits.open(cube_path, memmap=True) as hdul:
            data = hdul['DATA'].data
            mask = hdul['MASK'].data
        sat   = (mask[0] & sat_bit) != 0
        band  = np.nanmean(data[200:800].astype(float), axis=0)
        results.append((ifuslot, band, sat, sat.sum()))
    return results

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for shot_i, (shotid, label) in enumerate([(GOOD_SHOT, 'GOOD'), (BAD_SHOT, 'BAD')]):
    res = quick_sat_image(shotid, shot_groups, pdr_dir)
    for j, (ifuslot, band, sat, nsat) in enumerate(res[:4]):
        ax_im  = axes[shot_i][j]
        finite = band[np.isfinite(band)]
        vmin, vmax = np.nanpercentile(finite, [1, 99]) if finite.size else (0, 1)
        ax_im.imshow(band, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
        ax_im.contour(sat.astype(float), levels=[0.5], colors='red', linewidths=0.8)
        ax_im.set_title(f'{label} {shotid}\n{ifuslot}  {nsat}spx', fontsize=8)
        ax_im.set_xticks([]); ax_im.set_yticks([])
plt.tight_layout()
plt.show()


# ── CELL 3: Check whether SAT pixels are morphologically linear ──────────────
# Fit a line to SAT pixel positions and compute residuals
# A real streak should have very small residuals; scattered pixels will not

from numpy.linalg import svd

def streak_linearity(shotid, shot_groups, pdr_dir, sat_bit=1024):
    rows = shot_groups.get(shotid, [])
    for row in rows:
        ifuslot   = str(row['ifuslot']).strip()
        cube_path = op.join(pdr_dir, 'datacubes', str(shotid),
                            f'dex_cube_{shotid}_{ifuslot}.fits')
        if not op.exists(cube_path):
            continue
        with fits.open(cube_path, memmap=True) as hdul:
            mask = hdul['MASK'].data
        sat = (mask[0] & sat_bit) != 0
        if not sat.any():
            continue
        yy, xx = np.where(sat)
        pts = np.column_stack([xx - xx.mean(), yy - yy.mean()])
        if len(pts) < 2:
            continue
        _, s, _ = svd(pts, full_matrices=False)
        # ratio of 1st to 2nd singular value: high = linear, low = scattered
        linearity = s[0] / s[1] if s[1] > 0 else np.inf
        # RMS perpendicular residual from the principal axis (pixels)
        _, _, vt = svd(pts, full_matrices=False)
        perp = pts @ vt[1]   # projection onto minor axis
        rms_perp = np.sqrt(np.mean(perp**2))
        print(f"  Shot {shotid}  IFU {ifuslot}:  n_sat={sat.sum():5d}  "
              f"linearity={linearity:7.1f}  rms_perp={rms_perp:.2f} px")

print("Good shot:")
streak_linearity(GOOD_SHOT, shot_groups, pdr_dir)
print("\nBad shots:")
for sid in [20240314026, 20240305021, 20230920016, 20230920017, 20230920018]:
    if sid in shot_groups:
        streak_linearity(sid, shot_groups, pdr_dir)


# ── CELL 4: Check if the issue is date-related ───────────────────────────────
# Compute linearity score for all shots and plot vs date

print("Computing linearity for all shots (may take a minute)...")
linearity_results = []
for shotid in all_shotids:
    rows = shot_groups[shotid]
    date = int(str(shotid)[:8])
    for row in rows:
        ifuslot   = str(row['ifuslot']).strip()
        cube_path = op.join(pdr_dir, 'datacubes', str(shotid),
                            f'dex_cube_{shotid}_{ifuslot}.fits')
        if not op.exists(cube_path):
            continue
        try:
            with fits.open(cube_path, memmap=True) as hdul:
                mask = hdul['MASK'].data
        except Exception:
            continue
        sat = (mask[0] & SAT_BIT_CHECK) != 0
        if sat.sum() < 5:
            continue
        yy, xx = np.where(sat)
        pts = np.column_stack([xx - xx.mean(), yy - yy.mean()])
        _, s, _ = svd(pts, full_matrices=False)
        lin = s[0] / s[1] if s[1] > 0 else np.inf
        linearity_results.append((date, shotid, ifuslot, int(sat.sum()), float(lin)))

dates, sids, ifus, nspax, lins = zip(*linearity_results)
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
sc = axes[0].scatter(dates, lins, c=lins, cmap='RdYlGn', vmin=1, vmax=50,
                     s=10, alpha=0.7)
axes[0].axhline(10, color='k', ls='--', lw=0.8, label='linearity=10 threshold')
axes[0].set_ylabel('Linearity (SVD ratio)')
axes[0].set_title('SAT mask linearity vs observation date\n(high = streak-like, low = scattered)')
axes[0].set_ylim(0, 80)
axes[0].legend(fontsize=8)
plt.colorbar(sc, ax=axes[0])

axes[1].scatter(dates, nspax, s=8, alpha=0.6, color='C0')
axes[1].set_ylabel('N SAT spaxels')
axes[1].set_xlabel('Date (YYYYMMDD)')
axes[1].set_title('SAT spaxel count vs date')
# mark the suspect date range
for ax in axes:
    ax.axvspan(20230901, 20240430, color='orange', alpha=0.15, label='suspect range')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('sat_mask_linearity_vs_date.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved sat_mask_linearity_vs_date.png")